In [57]:
!pip install brkraw==0.3.11


In [6]:
import sys

REPO_PATH = r"S:\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\resomapper_dev-main\resomapper_dev-main"

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)


In [3]:
import sys
import os
import shutil
from glob import glob

# =========================
# 1) CONFIGURACIÓN 
# =========================

# Carpeta del repo donde está resomapper
REPO_PATH = r"S:\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\resomapper_dev-main\resomapper_dev-main"

# Carpeta donde está tu estudio 
ROOT_INPUT = r"S:\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times\R87\sourcedata\sub-H250924_R86_tiempdifusion_2_250924\dwi"

# Carpeta de salida
OUTPUT_DIR = r"S:\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times\R87\sourcedata\sub-H250924_R86_tiempdifusion_2_250924\preproc\dwi"


# =========================
# 2) VALIDACIONES BÁSICAS
# =========================
if not os.path.isdir(REPO_PATH):
    raise FileNotFoundError(f"REPO_PATH no existe: {REPO_PATH}")

if not os.path.isdir(ROOT_INPUT):
    raise FileNotFoundError(f"ROOT_INPUT no existe: {ROOT_INPUT}")

# Asegura imports desde el repo
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print("? Repo:", REPO_PATH)
print("? Input root:", ROOT_INPUT)
print("? Output:", OUTPUT_DIR)


? Repo: S:\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\resomapper_dev-main\resomapper_dev-main
? Input root: S:\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times\R87\sourcedata\sub-H250924_R86_tiempdifusion_2_250924\dwi
? Output: S:\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times\R87\sourcedata\sub-H250924_R86_tiempdifusion_2_250924\preproc\dwi


In [ ]:
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Ahora que ya añadimos la ruta, los IMPORTS funcionarán:
from resomapper_preprocessing import denoise, gibbs_suppress, n4_bias_field_correct
from resomapper_bruker_conversion import convert_single_study_bruker
from resomapper_masking import manual_mask
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.ndimage import rotate

In [1]:
from resomapper_preprocessing import denoise, gibbs_suppress, n4_bias_field_correct
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.ndimage import rotate
from resomapper_masking import manual_mask 

ModuleNotFoundError: No module named 'resomapper_preprocessing'

In [1]:

# =========================
# 3) ENCONTRAR CARPETA BRUKER 
# ========================

def find_bruker_study_folder(root):
    for dirpath, dirnames, filenames in os.walk(root):
        if "subject" in dirnames:
            return dirpath
        if "subject" in filenames:
            return dirpath
    return None

bruker_study_path = find_bruker_study_folder(ROOT_INPUT)
print("?? Carpeta Bruker detectada:", bruker_study_path)

if bruker_study_path is None:
    raise ValueError(
    )

try:
    top = os.listdir(bruker_study_path)
    print("?? 'subject' está en la carpeta?:", "subject" in top)
    print("?? Contenido (top 20):", top[:20])
except Exception as e:
    print("?? No pude listar contenido de la carpeta detectada:", e)


# =========================
# 4) CONVERTIR SOLO ESE ESTUDIO
# =========================
from resomapper_bruker_conversion import convert_single_study_bruker

os.makedirs(OUTPUT_DIR, exist_ok=True)

subj_sess_name = convert_single_study_bruker(bruker_study_path, OUTPUT_DIR)

if subj_sess_name is None:

    raise RuntimeError(

    )

print("\n? Conversión terminada.")
print("?? Nombre sujeto/sesión:", subj_sess_name)

# ========================
# 5) MOSTRAR RESUMEN DE SALIDA
# =========================

out_subject_dir = os.path.join(OUTPUT_DIR, "sourcedata", subj_sess_name)
print("\n?? Carpeta de salida:", out_subject_dir)

if not os.path.isdir(out_subject_dir):
    print("?? No encuentro la carpeta de salida esperada.")
else:
    for root, dirs, files in os.walk(out_subject_dir):
        rel = os.path.relpath(root, out_subject_dir)
        print(f"\n[{rel}]")
        if dirs:
            print("  dirs :", dirs)
        if files:
            preview = files[:15]
            print("  files:", preview, ("..." if len(files) > 15 else ""))

NameError: name 'ROOT_INPUT' is not defined

In [62]:
# =========================
# 6) PREPROC AUTOMÁTICO SOLO DWI/DTI ? DERIVATIVES
# =========================

DERIV_ROOT = os.path.join(OUTPUT_DIR, "derivatives")
DERIV_DWI  = os.path.join(DERIV_ROOT, "dwi")
os.makedirs(DERIV_DWI, exist_ok=True)

dwi_dir = os.path.join(out_subject_dir, "dwi")
if not os.path.isdir(dwi_dir):
    raise FileNotFoundError(f"No existe la carpeta dwi/: {dwi_dir}")

dwi_files = sorted(glob(os.path.join(dwi_dir, "*_dwi.nii.gz")))

if len(dwi_files) == 0:
    raise FileNotFoundError(f"No encontré *_dwi.nii.gz en: {dwi_dir}")
    
print("\n?? DWI encontrados:")
for f in dwi_files:
    print("  -", os.path.basename(f))

# =======================
# AJUSTES
# =======================

DO_DENOISE = True
DO_GIBBS   = True
DO_N4      = False

DENOISE_FILTER = "p"  # Patch2Self

# =========================
# PROCESAR
# =========================

for nifti_path in dwi_files:
    base = os.path.basename(nifti_path).replace(".nii.gz", "")
    print("\n?? Preprocesando:", base)

    # Copiar bval/bvec
    for ext in [".bval", ".bvec"]:
        src = nifti_path.replace(".nii.gz", ext)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DERIV_DWI, os.path.basename(src)))

    current_path = nifti_path
    # 1) DENOISE
    if DO_DENOISE:
        _, current_path, used_filter = denoise(
            current_path,
            modality="DTI",
            output_folder=DERIV_DWI,
            params="default",
            selected_filter=DENOISE_FILTER
        )
        print("   ? Denoise:", used_filter)
    # 2) GIBBS
    if DO_GIBBS:
        _, current_path = gibbs_suppress(
            current_path,
            unringed_nii_output_path=os.path.join(DERIV_DWI, base + "_preproc.nii.gz"),
            check_params=False,
        )
        print("   ? Gibbs")
    # 3) N4 
    if DO_N4:
        _, current_path, _ = n4_bias_field_correct(
            current_path,
            corrected_nii_output_path=os.path.join(DERIV_DWI, base + "_preproc.nii.gz"),
            params=None,
            check_params=False,
        )
        print("   ? N4")

    final_path = os.path.join(DERIV_DWI, base + "_preproc.nii.gz")
    if os.path.abspath(current_path) != os.path.abspath(final_path):
        shutil.copy2(current_path, final_path)
    print("? FINAL ->", os.path.basename(final_path))

print("\n?? Preprocesado terminado.")
print("?? Resultados en:", DERIV_DWI)



?? DWI encontrados:
  - sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-12_run-1_dwi.nii.gz
  - sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-15_run-1_dwi.nii.gz
  - sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-18_run-1_dwi.nii.gz
  - sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-8_run-1_dwi.nii.gz

?? Preprocesando: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-12_run-1_dwi

    ... applying p2s denoising filter


   ? Denoise: p
   ? Gibbs
? FINAL -> sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-12_run-1_dwi_preproc.nii.gz

?? Preprocesando: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-15_run-1_dwi

    ... applying p2s denoising filter


   ? Denoise: p
   ? Gibbs
? FINAL -> sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-15_run-1_dwi_preproc.nii.gz

?? Preprocesando: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-18_run-1_dwi

    ... applying p2s denoising filter


   ? Denoise: p
   ? Gibbs
? FINAL -> sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-18_run-1_dwi_preproc.nii.gz

?? Preprocesando: sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-8_run-1_dwi

    ... applying p2s denoising filter


   ? Denoise: p
   ? Gibbs
? FINAL -> sub-G230924_R107_Tiemposdifusion_2_2_230924_acq-8_run-1_dwi_preproc.nii.gz

?? Preprocesado terminado.
?? Resultados en: C:\Users\Estudiantes\Desktop\Estudio_tiemposdedifusion\R107\derivatives\dwi


In [2]:
# =========================
# 7) CREATE MASK
# =========================

def _as_manualmask_view(slice2d):
    out = rotate(slice2d, 270, reshape=False)
    out = np.flip(out, axis=1)
    return out

def preview_mask_inline_5slices_oriented_like_manual(image_path, mask_path, title="", vol_index=0, slices=None):
    img_nii = nib.load(image_path)
    msk_nii = nib.load(mask_path)

    img = img_nii.get_fdata()
    msk = msk_nii.get_fdata()

    img3 = img[..., vol_index] if img.ndim == 4 else img
    msk3 = msk[..., 0] if msk.ndim == 4 else msk

    Z = img3.shape[2]
    if slices is None:
        sel = list(range(Z)) if Z <= 5 else np.linspace(0, Z - 1, 5).round().astype(int).tolist()
    else:
        sel = list(slices)[:5]

    img3n = img3.astype(np.float32)
    img3n -= img3n.min()
    mx = img3n.max()
    if mx > 0:
        img3n /= mx

    fig, axes = plt.subplots(1, len(sel), figsize=(3.2 * len(sel), 3.2))
    if len(sel) == 1:
        axes = [axes]

    for ax, z in zip(axes, sel):
        im2d = img3n[:, :, z]
        mk2d = (msk3[:, :, z] > 0).astype(np.uint8)

        im2d_v = _as_manualmask_view(im2d)
        mk2d_v = _as_manualmask_view(mk2d)

        ax.imshow(im2d_v, cmap="gray", origin="upper")  # aquí ya no hace falta .T
        ax.contour(mk2d_v, levels=[0.5], colors="r", linewidths=1)
        ax.set_title(f"z={z}")
        ax.axis("off")
        
    fig.suptitle(title + (f" | vol {vol_index}" if img.ndim == 4 else ""), y=1.02)
    plt.tight_layout()
    plt.show()

def ask_yes_no_text(prompt, default="y"):
    default = default.lower()
    suf = " [Y/n] " if default == "y" else " [y/N] "
    ans = input(prompt + suf).strip().lower()
    if ans == "":
        ans = default
    return ans in ("y", "yes", "s", "si", "sí")

In [3]:
def manual_mask_with_preview(nifti_path, mask_path, title=""):
    while True:
        print("\n??? Dibuja la máscara (ventana OpenCV):")
        manual_mask(nifti_path, mask_path, ask_if_repeat=False)
        
        preview_mask_inline_5slices_oriented_like_manual(nifti_path, mask_path, title=title, vol_index=0)

        ok = ask_yes_no_text("¿La máscara está bien?", default="y")
        if ok:
            return True
        else:
            print("?? Repetimos el dibujo...")

In [4]:
def apply_mask_and_save(image_path, mask_path, masked_output_path):
    img_nii = nib.load(image_path)
    msk_nii = nib.load(mask_path)

    img = img_nii.get_fdata()
    msk = msk_nii.get_fdata()

    if img.ndim == 4 and msk.ndim == 3:
        masked = img * msk[..., np.newaxis]
    else:
        masked = img * msk
        
    out = nib.Nifti1Image(masked.astype(np.float32), img_nii.affine, img_nii.header)
    nib.save(out, masked_output_path)

In [5]:

# =========================
# T2w MASK -> guardar en anat/
# =========================
anat_dir = os.path.join(out_subject_dir, "anat")
anat_niis = sorted(glob(os.path.join(anat_dir, "*.nii.gz")))

if not anat_niis:
    print("No hay NIfTI en anat/")
else:

    t2_candidates = [p for p in anat_niis if "t2w" in os.path.basename(p).lower()]
    t2w_path = t2_candidates[0] if t2_candidates else anat_niis[0]

    base = os.path.basename(t2w_path).replace(".nii.gz", "")
    t2_mask   = os.path.join(anat_dir, base + "_mask.nii.gz")
    t2_masked = os.path.join(anat_dir, base + "_masked.nii.gz")

    manual_mask_with_preview(t2w_path, t2_mask, title=os.path.basename(t2w_path))
    apply_mask_and_save(t2w_path, t2_mask, t2_masked)

    print("? T2w mask   ->", t2_mask)
    print("? T2w masked ->", t2_masked)


# =========================
# DWI Mask-> guardar en preproc/dwi/
# =========================

preproc_dwi_dir = os.path.join(out_subject_dir, "preproc", "dwi")
raw_dwi_dir     = os.path.join(out_subject_dir, "dwi")
os.makedirs(preproc_dwi_dir, exist_ok=True)

dwi_files = sorted(glob(os.path.join(preproc_dwi_dir, "*_preproc.nii.gz")))
if not dwi_files:
    dwi_files = sorted(glob(os.path.join(raw_dwi_dir, "*_dwi.nii.gz")))

if not dwi_files:
    print("?? No encontré DWI ni en preproc/dwi/ ni en dwi/")
else:

    dwi_path = dwi_files[0]
    base = os.path.basename(dwi_path).replace(".nii.gz", "")
    dwi_mask   = os.path.join(preproc_dwi_dir, base + "_mask.nii.gz")
    dwi_masked = os.path.join(preproc_dwi_dir, base + "_masked.nii.gz")

    manual_mask_with_preview(dwi_path, dwi_mask, title=os.path.basename(dwi_path))
    apply_mask_and_save(dwi_path, dwi_mask, dwi_masked)

    print("? DWI mask   ->", dwi_mask)
    print("? DWI masked ->", dwi_masked)



NameError: name 'os' is not defined

In [1]:
import os
from glob import glob
import nibabel as nib

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS (R87 con DKI_4_1)
# ==========================================
ROOT_ESTUDIO = r"S:\mdnovalbos\info_services\blizarbe_lab_share\TFG Lola\NEXI\Diffusion_times"
ID_RATON = "R87"
ID_TIEMPO = "DKI_4_1" 

rat_path = os.path.join(ROOT_ESTUDIO, ID_RATON)

# ==========================================
# 2. LOCALIZACIÓN DE ARCHIVOS
# ==========================================
deriv_dwi_path = os.path.join(rat_path, "derivatives", "dwi")
dwi_corr_glob = glob(os.path.join(deriv_dwi_path, f"*{ID_TIEMPO}*_preproc.nii.gz"))
sourcedata_glob = glob(os.path.join(rat_path, "sourcedata", "sub-*", "preproc", "dwi"))

if not dwi_corr_glob:
    print(f"❌ ERROR: No se encontró archivo con '{ID_TIEMPO}' en: {deriv_dwi_path}")
elif not sourcedata_glob:
    print(f"❌ ERROR: No se encontró la carpeta de destino en sourcedata")
else:
    dwi_path = dwi_corr_glob[0]
    mask_dest_dir = sourcedata_glob[0]
    
    base_name = os.path.basename(dwi_path).replace(".nii.gz", "")
    final_mask_path = os.path.join(mask_dest_dir, base_name + "_mask.nii.gz")
    final_masked_path = os.path.join(mask_dest_dir, base_name + "_masked.nii.gz")

    print(f"✅ Imagen cargada: {os.path.basename(dwi_path)}")
    
    # ==========================================
    # 3. LANZAR DIBUJO
    # ==========================================
    manual_mask_with_preview(dwi_path, final_mask_path, title=f"CORRECCIÓN {ID_RATON} - {ID_TIEMPO}")
    apply_mask_and_save(dwi_path, final_mask_path, final_masked_path)

    print(f"\n🎉 Nueva máscara guardada en: {mask_dest_dir}")

ModuleNotFoundError: No module named 'nibabel'